> **Save Your Work** — Before you begin, click **File → Save a copy in Drive** (Google Colab) or **File → Download** so you do not lose your progress.

# Module 9 Assessment — Machine Learning: Classification

Build and compare binary classifiers on the mushroom dataset.

## Dataset

The mushroom dataset contains physical descriptions of mushrooms labeled as edible (e=0) or poisonous (p=1). This is a safety-critical classification problem: a false negative (predicting edible when actually poisonous) has severe consequences.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_auc_score
)
import matplotlib.pyplot as plt

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/mushroom/agaricus-lepiota.data"
columns = [
    "class", "cap_shape", "cap_surface", "cap_color", "bruises",
    "odor", "gill_attachment", "gill_spacing", "gill_size", "gill_color",
    "stalk_shape", "stalk_root", "stalk_surface_above", "stalk_surface_below",
    "stalk_color_above", "stalk_color_below", "veil_type", "veil_color",
    "ring_number", "ring_type", "spore_print_color", "population", "habitat"
]
df = pd.read_csv(url, header=None, names=columns)
print(df.shape)                    # (8124, 23)
print(df["class"].value_counts())
# e    4208
# p    3916

## Task 1: Data Preparation

1. Encode the target: `label = 1` if poisonous, `0` if edible
2. Label-encode all feature columns
3. Perform a stratified 80/20 train/test split (`random_state=42`)
4. Print train and test shapes

In [ ]:
# Encode target
df["label"] = (df["class"] == "p").astype(int)

X = df.drop(columns=["class", "label"])
y = df["label"]

# Encode all categorical feature columns
for col in X.columns:
    X[col] = LabelEncoder().fit_transform(X[col])

# Train/test split (stratified to preserve class balance)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("X_train shape:", X_train.shape)  # (6499, 22)
print("X_test shape: ", X_test.shape)   # (1625, 22)
print("y_train distribution:")
print(y_train.value_counts())
# 0    3366
# 1    3133

## Task 2: Logistic Regression Baseline

Train a `LogisticRegression(max_iter=1000)`. Print the full classification report. Note which metric matters most for this safety-critical domain and why (write 1 sentence in the markdown cell below).

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

print("Logistic Regression Classification Report:")
print(classification_report(y_test, y_pred_lr, target_names=["edible", "poisonous"]))
# Typical output:
#               precision    recall  f1-score   support
#       edible       0.97      0.99      0.98       842
#    poisonous       0.99      0.97      0.98       783
#     accuracy                           0.98      1625

lr_auc = roc_auc_score(y_test, lr.predict_proba(X_test)[:, 1])
print(f"AUC: {lr_auc:.4f}")  # ~0.9994

In this safety-critical domain, **recall for the poisonous class** (label=1) is the most important metric, because a false negative — predicting a poisonous mushroom as edible — could result in someone eating a toxic mushroom, which is far more dangerous than a false positive that causes someone to discard an edible mushroom.

## Task 3: Decision Tree

Train a `DecisionTreeClassifier(max_depth=5, random_state=42)`. Display the confusion matrix. Compare false negatives (predicting edible when poisonous) with logistic regression.

In [ ]:
dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

print("Decision Tree Classification Report:")
print(classification_report(y_test, y_pred_dt, target_names=["edible", "poisonous"]))
# Typical output:
#               precision    recall  f1-score   support
#       edible       1.00      0.99      0.99       842
#    poisonous       0.99      1.00      0.99       783
#     accuracy                           0.99      1625

dt_auc = roc_auc_score(y_test, dt.predict_proba(X_test)[:, 1])
print(f"AUC: {dt_auc:.4f}")  # ~0.9990

# Display confusion matrix
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_lr,
    display_labels=["edible", "poisonous"],
    ax=axes[0]
)
axes[0].set_title("Logistic Regression (threshold=0.5)")

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_dt,
    display_labels=["edible", "poisonous"],
    ax=axes[1]
)
axes[1].set_title("Decision Tree (max_depth=5)")

plt.tight_layout()
plt.show()
# Decision Tree tends to produce fewer false negatives than LR on this dataset
# because odor and spore_print_color are near-perfect separators at low tree depth

## Task 4: Threshold Adjustment

For the logistic regression model, lower the decision threshold to 0.3. Print the classification report. Answer in the markdown cell: does recall for the poisonous class improve? What happens to precision?

In [ ]:
# Get probability predictions from logistic regression
y_proba = lr.predict_proba(X_test)[:, 1]
y_pred_adjusted = (y_proba >= 0.3).astype(int)

print("Logistic Regression (threshold=0.3) Classification Report:")
print(classification_report(y_test, y_pred_adjusted, target_names=["edible", "poisonous"]))
# Lowering the threshold flags more mushrooms as poisonous.
# Recall for poisonous class typically increases to ~0.99–1.00
# Precision for poisonous class typically decreases slightly (more false positives)

lr_adj_auc = roc_auc_score(y_test, y_proba)
print(f"AUC (unchanged — same model): {lr_adj_auc:.4f}")
# AUC does not change with threshold — it reflects the full ROC curve

Yes, lowering the threshold to 0.3 improves recall for the poisonous class: the model now flags mushrooms as poisonous with less certainty, so fewer poisonous mushrooms are missed (fewer false negatives). The tradeoff is that precision for the poisonous class decreases — some edible mushrooms are now incorrectly predicted as poisonous (more false positives), which is an acceptable cost in this safety-critical context.

## Task 5: Comparison Table

Fill in this table using your results:

| Model | Accuracy | Precision (poisonous) | Recall (poisonous) | F1 (poisonous) | AUC |
|-------|----------|-----------------------|--------------------|----------------|-----|
| Logistic Regression (threshold=0.5) | ~0.98 | ~0.99 | ~0.97 | ~0.98 | ~0.9994 |
| Logistic Regression (threshold=0.3) | ~0.97 | ~0.96 | ~1.00 | ~0.98 | ~0.9994 |
| Decision Tree (max_depth=5) | ~0.99 | ~0.99 | ~1.00 | ~0.99 | ~0.9990 |

*Note: Exact values depend on your run. AUC does not change when you adjust the threshold — it characterizes the model's full ROC curve independent of any single cutoff.*

## Task 6: Recommendation

In the markdown cell below, write 2–3 sentences justifying which model and threshold you would deploy in a real application that helps hikers identify safe mushrooms.

For a hiker safety application, I would deploy the **Decision Tree (max_depth=5)** or the **Logistic Regression with threshold=0.3**, both of which achieve near-perfect recall on the poisonous class. The asymmetric cost of errors makes recall the dominant metric: a false negative (telling a hiker a poisonous mushroom is safe) could be fatal, whereas a false positive (flagging an edible mushroom as dangerous) only results in the hiker skipping a meal. Given this priority, I would pair whichever model achieves recall=1.00 for poisonous with a clear user warning that the app is not a substitute for expert mycological identification, and I would continuously monitor for distribution shifts as new mushroom varieties are encountered.